In [31]:
import pandas as pd
import numpy as np

In [32]:
df = pd.read_csv("./../Data/student_data.csv")
df.head()

,Hours Studied,Previous Scores,Extracurricular Activities,Duration of Sleep,Sample Question Papers Practiced,Performance
0,7,99,Yes,9,1,91
1,4,82,No,4,2,65
2,8,51,Yes,7,2,45
3,5,52,Yes,5,2,36
4,7,75,No,8,5,66


In [33]:
##shuffle data
df = df.sample(frac=1)
df.head()

,Hours Studied,Previous Scores,Extracurricular Activities,Duration of Sleep,Sample Question Papers Practiced,Performance
3503,6,42,No,8,6,29
8184,1,63,Yes,8,3,37
1743,1,51,No,4,4,21
6390,4,58,No,5,1,39
2358,4,77,Yes,7,9,61


In [34]:
#### Now Test Train Split
train_last = int(0.70*len(df))
train_last

7000

In [35]:
train = df.iloc[0:train_last,:]
test = df.iloc[train_last:len(df),:]
len(train),len(test)

(7000, 3000)

In [36]:
###training values
train_ohe = pd.get_dummies(train,columns=["Extracurricular Activities"],drop_first=False,dtype=int)
columns = train_ohe.columns
train_ohe.head(),columns

(      Hours Studied  Previous Scores  Duration of Sleep  \
 3503              6               42                  8   
 8184              1               63                  8   
 1743              1               51                  4   
 6390              4               58                  5   
 2358              4               77                  7   
 
       Sample Question Papers Practiced  Performance  \
 3503                                 6           29   
 8184                                 3           37   
 1743                                 4           21   
 6390                                 1           39   
 2358                                 9           61   
 
       Extracurricular Activities_No  Extracurricular Activities_Yes  
 3503                              1                               0  
 8184                              0                               1  
 1743                              1                               0  
 6390           

In [37]:
xcolumns = [column for column in train_ohe.columns if column != "Performance"]
xcolumns

['Hours Studied',
 'Previous Scores',
 'Duration of Sleep',
 'Sample Question Papers Practiced',
 'Extracurricular Activities_No',
 'Extracurricular Activities_Yes']

In [38]:
ycolumns = ["Performance"]
ycolumns

['Performance']

In [39]:
x_tr = train_ohe.loc[:,xcolumns]
y_tr = train_ohe.loc[:,ycolumns]

In [40]:
saved_columns = x_tr.columns
saved_columns

Index(['Hours Studied', 'Previous Scores', 'Duration of Sleep',
       'Sample Question Papers Practiced', 'Extracurricular Activities_No',
       'Extracurricular Activities_Yes'],
      dtype='object')

In [41]:
#### training
folds = 5
alphas = [0.001,0.01,0.1,1]
lambdas = np.logspace(-6,6,50)

In [42]:
lambdas

array([1.00000000e-06, 1.75751062e-06, 3.08884360e-06, 5.42867544e-06,
       9.54095476e-06, 1.67683294e-05, 2.94705170e-05, 5.17947468e-05,
       9.10298178e-05, 1.59985872e-04, 2.81176870e-04, 4.94171336e-04,
       8.68511374e-04, 1.52641797e-03, 2.68269580e-03, 4.71486636e-03,
       8.28642773e-03, 1.45634848e-02, 2.55954792e-02, 4.49843267e-02,
       7.90604321e-02, 1.38949549e-01, 2.44205309e-01, 4.29193426e-01,
       7.54312006e-01, 1.32571137e+00, 2.32995181e+00, 4.09491506e+00,
       7.19685673e+00, 1.26485522e+01, 2.22299648e+01, 3.90693994e+01,
       6.86648845e+01, 1.20679264e+02, 2.12095089e+02, 3.72759372e+02,
       6.55128557e+02, 1.15139540e+03, 2.02358965e+03, 3.55648031e+03,
       6.25055193e+03, 1.09854114e+04, 1.93069773e+04, 3.39322177e+04,
       5.96362332e+04, 1.04811313e+05, 1.84206997e+05, 3.23745754e+05,
       5.68986603e+05, 1.00000000e+06])

In [69]:
folds_inputs = []
folds_outputs = []
for i in range(0,len(x_tr),foldsize):
    temp = x_tr.iloc[i:i+foldsize]
    folds_inputs.append(temp)
    temp = y_tr.iloc[i:i+foldsize]
    folds_outputs.append(temp)
for m in folds_inputs:
    print(len(m))
for m in folds_outputs:
    print(len(m))

1400
1400
1400
1400
1400
1400
1400
1400
1400
1400


In [70]:
folds_inputs[2].head()

,Hours Studied,Previous Scores,Duration of Sleep,Sample Question Papers Practiced,Extracurricular Activities_No,Extracurricular Activities_Yes
1543,7,46,5,2,1,0
3551,6,77,8,0,0,1
2196,5,99,9,0,1,0
5790,7,79,7,5,0,1
5555,5,56,8,2,0,1


In [78]:
r2_tuned = []
alpha_lambda = []
for alpha in alphas:
    for l in lambdas:
        r2scores = []
        for k in range(0,5):
            train_inputs = [folds_inputs[i] for i in range(len(folds_inputs)) if i!=k]
            train_outputs = [folds_outputs[i] for i in range(len(folds_outputs)) if i!=k]
            val_inputs = folds_inputs[k]
            val_outputs = folds_outputs[k]
            x_train = pd.concat(train_inputs,axis=0)
            y_train = pd.concat(train_outputs,axis=0)
            mean_vals = {}
            std_vals = {}
            ## standardising
            for column in x_train.columns:
                mean = x_train[column].mean()
                std = x_train[column].std()
                mean_vals[column] = mean
                std_vals[column] = std
                x_train[column] = (x_train[column]-mean)/std

            ##training
            betas = np.zeros(x_train.shape[1]+1)
            x_tr1 = np.c_[np.ones(len(x_train)),np.array(x_train)]
            y_tr1 = np.array(y_train).ravel()
            for i in range(0,1000):       
                predictions = np.matmul(x_tr1,betas)
                gradient = np.matmul(np.transpose(x_tr1),predictions-y_tr1)
                identity_mat = np.identity(len(betas))
                d = np.identity(len(betas))
                d[0][0] = 0
                shrinkage = np.matmul((identity_mat-alpha*l*d),betas)
                n = len(x_tr1)
                ##update
                betas = shrinkage - 1/n*alpha*gradient
            ##validation
            ## standardising
            x_vals = val_inputs.copy()
            for column in val_inputs.columns:
                x_vals[column] = (x_vals[column]-mean_vals[column])/std_vals[column]
            ##evaluating
            x_v1 = np.c_[np.ones(len(x_vals)),np.array(x_vals)]
            y_v1 = np.array(val_outputs).ravel()
            predictions = np.matmul(x_v1,betas)
            errors = predictions - y_v1
            r2_score = 1 - np.sum(errors**2)/(np.sum((y_v1-np.mean(y_v1))**2))
            r2scores.append(r2_score)
        average = np.mean(np.array(r2scores))
        print("Alpha: "+str(alpha)+ " Lmabda: "+str(l)+ " R2: "+str(average))
        alpha_lambda.append((alpha,l))
        r2_tuned.append(average)

Alpha: 0.001 Lmabda: 1e-06 R2: -0.2738120282240976
Alpha: 0.001 Lmabda: 1.757510624854793e-06 R2: -0.27381217421847814
Alpha: 0.001 Lmabda: 3.0888435964774785e-06 R2: -0.273812430805201
Alpha: 0.001 Lmabda: 5.428675439323859e-06 R2: -0.27381288175914126
Alpha: 0.001 Lmabda: 9.540954763499944e-06 R2: -0.2738136743157919
Alpha: 0.001 Lmabda: 1.67683293681101e-05 R2: -0.27381506724332577
Alpha: 0.001 Lmabda: 2.94705170255181e-05 R2: -0.2738175153308334
Alpha: 0.001 Lmabda: 5.1794746792312125e-05 R2: -0.27382181787857923
Alpha: 0.001 Lmabda: 9.102981779915228e-05 R2: -0.27382937967641385
Alpha: 0.001 Lmabda: 0.00015998587196060574 R2: -0.27384266969204873
Alpha: 0.001 Lmabda: 0.0002811768697974231 R2: -0.2738660272690071
Alpha: 0.001 Lmabda: 0.0004941713361323833 R2: -0.2739070791785284
Alpha: 0.001 Lmabda: 0.000868511373751352 R2: -0.2739792305647141
Alpha: 0.001 Lmabda: 0.0015264179671752333 R2: -0.2741060442233713
Alpha: 0.001 Lmabda: 0.0026826957952797246 R2: -0.2743289415474258
Alpha:

C:\Users\Gaurish Garg\AppData\Local\Temp\ipykernel_14432\2882427221.py:29: RuntimeWarning: overflow encountered in matmul
  gradient = np.matmul(np.transpose(x_tr1),predictions-y_tr1)
C:\Users\Gaurish Garg\AppData\Local\Temp\ipykernel_14432\2882427221.py:29: RuntimeWarning: invalid value encountered in matmul
  gradient = np.matmul(np.transpose(x_tr1),predictions-y_tr1)
C:\Users\Gaurish Garg\AppData\Local\Temp\ipykernel_14432\2882427221.py:33: RuntimeWarning: invalid value encountered in matmul
  shrinkage = np.matmul((identity_mat-alpha*l*d),betas)


Alpha: 0.001 Lmabda: 3556.4803062231213 R2: nan


C:\Users\Gaurish Garg\AppData\Local\Temp\ipykernel_14432\2882427221.py:28: RuntimeWarning: invalid value encountered in matmul
  predictions = np.matmul(x_tr1,betas)


Alpha: 0.001 Lmabda: 6250.551925273976 R2: nan
Alpha: 0.001 Lmabda: 10985.411419875572 R2: nan
Alpha: 0.001 Lmabda: 19306.977288832455 R2: nan
Alpha: 0.001 Lmabda: 33932.217718953296 R2: nan
Alpha: 0.001 Lmabda: 59636.23316594637 R2: nan


...

Alpha: 0.001 Lmabda: 104811.3134154683 R2: nan
Alpha: 0.001 Lmabda: 184206.99693267164 R2: nan
Alpha: 0.001 Lmabda: 323745.754281764 R2: nan
Alpha: 0.001 Lmabda: 568986.6029018281 R2: nan
Alpha: 0.001 Lmabda: 1000000.0 R2: nan
Alpha: 0.01 Lmabda: 1e-06 R2: 0.9884314096108667
Alpha: 0.01 Lmabda: 1.757510624854793e-06 R2: 0.9884314095816469
Alpha: 0.01 Lmabda: 3.0888435964774785e-06 R2: 0.988431409527504
Alpha: 0.01 Lmabda: 5.428675439323859e-06 R2: 0.9884314094237319
Alpha: 0.01 Lmabda: 9.540954763499944e-06 R2: 0.9884314092147417
Alpha: 0.01 Lmabda: 1.67683293681101e-05 R2: 0.988431408765248
Alpha: 0.01 Lmabda: 2.94705170255181e-05 R2: 0.9884314077213944
Alpha: 0.01 Lmabda: 5.1794746792312125e-05 R2: 0.9884314051027318
Alpha: 0.01 Lmabda: 9.102981779915228e-05 R2: 0.9884313980788706
Alpha: 0.01 Lmabda: 0.00015998587196060574 R2: 0.9884313782565857
Alpha: 0.01 Lmabda: 0.0002811768697974231 R2: 0.9884313203316978
Alpha: 0.01 Lmabda: 0.0004941713361323833 R2: 0.9884311472737783
Alpha: 0.0

C:\Users\Gaurish Garg\AppData\Local\Temp\ipykernel_14432\2882427221.py:28: RuntimeWarning: overflow encountered in matmul
  predictions = np.matmul(x_tr1,betas)


Alpha: 0.1 Lmabda: 568986.6029018281 R2: nan
Alpha: 0.1 Lmabda: 1000000.0 R2: nan
Alpha: 1 Lmabda: 1e-06 R2: -59.76256662125542
Alpha: 1 Lmabda: 1.757510624854793e-06 R2: -59.854141190929774
Alpha: 1 Lmabda: 3.0888435964774785e-06 R2: -60.015418896796845
Alpha: 1 Lmabda: 5.428675439323859e-06 R2: -60.29990220418737
Alpha: 1 Lmabda: 9.540954763499944e-06 R2: -60.80310130221004
Alpha: 1 Lmabda: 1.67683293681101e-05 R2: -61.69750529962571
Alpha: 1 Lmabda: 2.94705170255181e-05 R2: -63.30089691021419
Alpha: 1 Lmabda: 5.1794746792312125e-05 R2: -66.21883601363629
Alpha: 1 Lmabda: 9.102981779915228e-05 R2: -71.67153964348114
Alpha: 1 Lmabda: 0.00015998587196060574 R2: -82.34788777392556
Alpha: 1 Lmabda: 0.0002811768697974231 R2: -105.05015692748562
Alpha: 1 Lmabda: 0.0004941713361323833 R2: -160.9381504127603
Alpha: 1 Lmabda: 0.000868511373751352 R2: -339.688232924456
Alpha: 1 Lmabda: 0.0015264179671752333 R2: -1257.2265876710087
Alpha: 1 Lmabda: 0.0026826957952797246 R2: -12474.76859269814
A

C:\Users\Gaurish Garg\AppData\Local\Temp\ipykernel_14432\2882427221.py:47: RuntimeWarning: overflow encountered in square
  r2_score = 1 - np.sum(errors**2)/(np.sum((y_v1-np.mean(y_v1))**2))


Alpha: 1 Lmabda: 0.42919342601287785 R2: -inf
Alpha: 1 Lmabda: 0.7543120063354607 R2: -inf
Alpha: 1 Lmabda: 1.325711365590108 R2: nan
Alpha: 1 Lmabda: 2.329951810515372 R2: nan
Alpha: 1 Lmabda: 4.094915062380427 R2: nan
Alpha: 1 Lmabda: 7.196856730011514 R2: nan
Alpha: 1 Lmabda: 12.648552168552959 R2: nan
Alpha: 1 Lmabda: 22.229964825261955 R2: nan
Alpha: 1 Lmabda: 39.06939937054613 R2: nan
Alpha: 1 Lmabda: 68.66488450042998 R2: nan
Alpha: 1 Lmabda: 120.67926406393264 R2: nan
Alpha: 1 Lmabda: 212.09508879201925 R2: nan
Alpha: 1 Lmabda: 372.7593720314938 R2: nan
Alpha: 1 Lmabda: 655.1285568595496 R2: nan
Alpha: 1 Lmabda: 1151.3953993264481 R2: nan
Alpha: 1 Lmabda: 2023.5896477251556 R2: nan
Alpha: 1 Lmabda: 3556.4803062231213 R2: nan
Alpha: 1 Lmabda: 6250.551925273976 R2: nan
Alpha: 1 Lmabda: 10985.411419875572 R2: nan
Alpha: 1 Lmabda: 19306.977288832455 R2: nan
Alpha: 1 Lmabda: 33932.217718953296 R2: nan
Alpha: 1 Lmabda: 59636.23316594637 R2: nan
Alpha: 1 Lmabda: 104811.3134154683 R2: 

In [79]:
alpha_lambda

[(0.001, 1e-06),
 (0.001, 1.757510624854793e-06),
 (0.001, 3.0888435964774785e-06),
 (0.001, 5.428675439323859e-06),
 (0.001, 9.540954763499944e-06),
 (0.001, 1.67683293681101e-05),
 (0.001, 2.94705170255181e-05),
 (0.001, 5.1794746792312125e-05),
 (0.001, 9.102981779915228e-05),
 (0.001, 0.00015998587196060574),
 (0.001, 0.0002811768697974231),
 (0.001, 0.0004941713361323833),
 (0.001, 0.000868511373751352),
 (0.001, 0.0015264179671752333),
 (0.001, 0.0026826957952797246),
 (0.001, 0.004714866363457394),
 (0.001, 0.008286427728546842),
 (0.001, 0.014563484775012445),
 (0.001, 0.025595479226995333),
 (0.001, 0.04498432668969444),
 (0.001, 0.07906043210907701),
 (0.001, 0.1389495494373136),
 (0.001, 0.244205309454865),
 (0.001, 0.42919342601287785),
 (0.001, 0.7543120063354607),
 (0.001, 1.325711365590108),
 (0.001, 2.329951810515372),
 (0.001, 4.094915062380427),
 (0.001, 7.196856730011514),
 (0.001, 12.648552168552959),
 (0.001, 22.229964825261955),
 (0.001, 39.06939937054613),
 (0.00

In [80]:
r2_tuned

[-0.2738120282240976,
 -0.27381217421847814,
 -0.273812430805201,
 -0.27381288175914126,
 -0.2738136743157919,
 -0.27381506724332577,
 -0.2738175153308334,
 -0.27382181787857923,
 -0.27382937967641385,
 -0.27384266969204873,
 -0.2738660272690071,
 -0.2739070791785284,
 -0.2739792305647141,
 -0.2741060442233713,
 -0.2743289415474258,
 -0.27472075004872465,
 -0.27540955186190963,
 -0.2766207080601014,
 -0.2787510101424082,
 -0.28249965774783214,
 -0.28909918734240564,
 -0.30071719863538166,
 -0.3211201854234137,
 -0.3565929697223284,
 -0.4164618694004412,
 -0.5105557187980087,
 -0.6391196804469278,
 -0.7804165750642642,
 -0.9016905329349422,
 -0.9882375074547218,
 -1.0442675072568328,
 -1.078661223294246,
 -1.0991140192642703,
 -1.1110512806822244,
 -1.117943212324589,
 -1.1218974473881667,
 -1.124158074765472,
 -1.125447830022145,
 -314232359722173.94,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 0.9884314096108667,
 0.9884314095816469,
 0.988431409527504,
 0.98843

In [82]:
r2_max_index = np.nanargmax(r2_tuned)
r2_max_index, r2_tuned[r2_max_index]

(106, 0.9884314291172123)

In [83]:
alpha = alpha_lambda[r2_max_index][0]
lamda = alpha_lambda[r2_max_index][1]

In [84]:
alpha,lamda

(0.1, 2.94705170255181e-05)

In [85]:
l = lamda
x_train = x_tr
y_train = y_tr
mean_vals = {}
std_vals = {}
## standardising
for column in x_train.columns:
    mean = x_train[column].mean()
    std = x_train[column].std()
    mean_vals[column] = mean
    std_vals[column] = std
    x_train[column] = (x_train[column]-mean)/std

##training
betas = np.zeros(x_train.shape[1]+1)
x_tr1 = np.c_[np.ones(len(x_train)),np.array(x_train)]
y_tr1 = np.array(y_train).ravel()
for i in range(0,1000):       
    predictions = np.matmul(x_tr1,betas)
    gradient = np.matmul(np.transpose(x_tr1),predictions-y_tr1)
    identity_mat = np.identity(len(betas))
    d = np.identity(len(betas))
    d[0][0] = 0
    shrinkage = np.matmul((identity_mat-alpha*l*d),betas)
    n = len(x_tr1)
    ##update
    betas = shrinkage - 1/n*alpha*gradient
train_predictions = np.matmul(x_tr1,betas)
errors = train_predictions - y_tr1
r2_score = 1 - np.sum(errors**2)/(np.sum((y_tr1-np.mean(y_tr1))**2))
print("Train_R2: "+str(r2_score))
            

Train_R2: 0.9884549113715533


In [94]:
##Now testing
test.head()

,Hours Studied,Previous Scores,Extracurricular Activities,Duration of Sleep,Sample Question Papers Practiced,Performance
1294,2,65,Yes,7,4,43
2242,5,45,No,7,2,33
898,8,60,No,4,7,50
3446,2,49,Yes,6,7,27
86,7,75,No,6,6,67


In [95]:
x_t = test.iloc[:,:-1]
y_t = test.iloc[:,-1]

In [96]:
x_t.head()

,Hours Studied,Previous Scores,Extracurricular Activities,Duration of Sleep,Sample Question Papers Practiced
1294,2,65,Yes,7,4
2242,5,45,No,7,2
898,8,60,No,4,7
3446,2,49,Yes,6,7
86,7,75,No,6,6


In [97]:
y_t.head()

1294    43
2242    33
898     50
3446    27
86      67
Name: Performance, dtype: int64

In [98]:
x_t = pd.get_dummies(x_t,columns=["Extracurricular Activities"],drop_first=False,dtype=int)
x_t.head()

,Hours Studied,Previous Scores,Duration of Sleep,Sample Question Papers Practiced,Extracurricular Activities_No,Extracurricular Activities_Yes
1294,2,65,7,4,0,1
2242,5,45,7,2,1,0
898,8,60,4,7,1,0
3446,2,49,6,7,0,1
86,7,75,6,6,1,0


In [99]:
x_t = x_t.reindex(columns=saved_columns)
x_t.head()

,Hours Studied,Previous Scores,Duration of Sleep,Sample Question Papers Practiced,Extracurricular Activities_No,Extracurricular Activities_Yes
1294,2,65,7,4,0,1
2242,5,45,7,2,1,0
898,8,60,4,7,1,0
3446,2,49,6,7,0,1
86,7,75,6,6,1,0


In [100]:
for column in x_t.columns:
    x_t[column] = (x_t[column]-mean_vals[column])/std_vals[column]
x_t.head()

,Hours Studied,Previous Scores,Duration of Sleep,Sample Question Papers Practiced,Extracurricular Activities_No,Extracurricular Activities_Yes
1294,-1.155321,-0.261173,0.279138,-0.201352,-1.016057,1.016057
2242,0.001874,-1.417754,0.279138,-0.896698,0.984056,-0.984056
898,1.159068,-0.550318,-1.489156,0.841666,0.984056,-0.984056
3446,-1.155321,-1.186438,-0.310293,0.841666,-1.016057,1.016057
86,0.773337,0.317118,-0.310293,0.493993,0.984056,-0.984056


In [101]:
betas

array([55.29828571,  7.38359663, 17.59412041,  0.82769822,  0.54649853,
       -0.15015466,  0.15015466])

In [102]:
x_t = np.c_[np.ones(len(x_t)),x_t]
x_t.shape

(3000, 7)

In [103]:
y_t = np.array(y_t).ravel()
y_t.shape

(3000,)

In [104]:
predictions = np.matmul(x_t,betas)
predictions.shape

(3000,)

In [105]:
errors = predictions - y_t
errors.shape

(3000,)

In [106]:
r2_score = 1 - np.sum(errors**2)/(np.sum((y_t-np.mean(y_t))**2))
r2_score

0.9894270003605294

In [107]:
alpha,lamda

(0.1, 2.94705170255181e-05)

In [109]:
lamdas = [alpha_lambda[k] for k in range(0,len(alpha_lambda))]
len(lamdas),len(r2_tuned)

(200, 200)

In [113]:
### tuning lambda only
r2_tuned = []
alpha_lambda = []
alpha = 0.001
for l in lambdas:
    r2scores = []
    for k in range(0,5):
        train_inputs = [folds_inputs[i] for i in range(len(folds_inputs)) if i!=k]
        train_outputs = [folds_outputs[i] for i in range(len(folds_outputs)) if i!=k]
        val_inputs = folds_inputs[k]
        val_outputs = folds_outputs[k]
        x_train = pd.concat(train_inputs,axis=0)
        y_train = pd.concat(train_outputs,axis=0)
        mean_vals = {}
        std_vals = {}
        ## standardising
        for column in x_train.columns:
            mean = x_train[column].mean()
            std = x_train[column].std()
            mean_vals[column] = mean
            std_vals[column] = std
            x_train[column] = (x_train[column]-mean)/std

        ##training
        betas = np.zeros(x_train.shape[1]+1)
        x_tr1 = np.c_[np.ones(len(x_train)),np.array(x_train)]
        y_tr1 = np.array(y_train).ravel()
        for i in range(0,10000):       
            predictions = np.matmul(x_tr1,betas)
            gradient = np.matmul(np.transpose(x_tr1),predictions-y_tr1)
            identity_mat = np.identity(len(betas))
            d = np.identity(len(betas))
            d[0][0] = 0
            shrinkage = np.matmul((identity_mat-alpha*l*d),betas)
            n = len(x_tr1)
            ##update
            betas = shrinkage - 1/n*alpha*gradient
        ##validation
        ## standardising
        x_vals = val_inputs.copy()
        for column in val_inputs.columns:
            x_vals[column] = (x_vals[column]-mean_vals[column])/std_vals[column]
        ##evaluating
        x_v1 = np.c_[np.ones(len(x_vals)),np.array(x_vals)]
        y_v1 = np.array(val_outputs).ravel()
        predictions = np.matmul(x_v1,betas)
        errors = predictions - y_v1
        r2_score = 1 - np.sum(errors**2)/(np.sum((y_v1-np.mean(y_v1))**2))
        r2scores.append(r2_score)
    average = np.mean(np.array(r2scores))
    print("Alpha: "+str(alpha)+ " Lmabda: "+str(l)+ " R2: "+str(average))
    alpha_lambda.append(l)
    r2_tuned.append(average)

Alpha: 0.001 Lmabda: 1e-06 R2: 0.9884314078800781
Alpha: 0.001 Lmabda: 1.757510624854793e-06 R2: 0.9884314078475759
Alpha: 0.001 Lmabda: 3.0888435964774785e-06 R2: 0.988431407787664
Alpha: 0.001 Lmabda: 5.428675439323859e-06 R2: 0.9884314076737537
Alpha: 0.001 Lmabda: 9.540954763499944e-06 R2: 0.9884314074469465
Alpha: 0.001 Lmabda: 1.67683293681101e-05 R2: 0.9884314069661428
Alpha: 0.001 Lmabda: 2.94705170255181e-05 R2: 0.9884314058672722
Alpha: 0.001 Lmabda: 5.1794746792312125e-05 R2: 0.988431403151951
Alpha: 0.001 Lmabda: 9.102981779915228e-05 R2: 0.9884313959583165
Alpha: 0.001 Lmabda: 0.00015998587196060574 R2: 0.9884313758379774
Alpha: 0.001 Lmabda: 0.0002811768697974231 R2: 0.9884313173902569
Alpha: 0.001 Lmabda: 0.0004941713361323833 R2: 0.9884311434165388
Alpha: 0.001 Lmabda: 0.000868511373751352 R2: 0.9884306178856507
Alpha: 0.001 Lmabda: 0.0015264179671752333 R2: 0.9884290171229722
Alpha: 0.001 Lmabda: 0.0026826957952797246 R2: 0.9884241213883959
Alpha: 0.001 Lmabda: 0.00471

C:\Users\Gaurish Garg\AppData\Local\Temp\ipykernel_14432\1031378910.py:30: RuntimeWarning: overflow encountered in matmul
  gradient = np.matmul(np.transpose(x_tr1),predictions-y_tr1)
C:\Users\Gaurish Garg\AppData\Local\Temp\ipykernel_14432\1031378910.py:30: RuntimeWarning: invalid value encountered in matmul
  gradient = np.matmul(np.transpose(x_tr1),predictions-y_tr1)
C:\Users\Gaurish Garg\AppData\Local\Temp\ipykernel_14432\1031378910.py:34: RuntimeWarning: invalid value encountered in matmul
  shrinkage = np.matmul((identity_mat-alpha*l*d),betas)


Alpha: 0.001 Lmabda: 3556.4803062231213 R2: nan


C:\Users\Gaurish Garg\AppData\Local\Temp\ipykernel_14432\1031378910.py:29: RuntimeWarning: invalid value encountered in matmul
  predictions = np.matmul(x_tr1,betas)


Alpha: 0.001 Lmabda: 6250.551925273976 R2: nan
Alpha: 0.001 Lmabda: 10985.411419875572 R2: nan
Alpha: 0.001 Lmabda: 19306.977288832455 R2: nan
Alpha: 0.001 Lmabda: 33932.217718953296 R2: nan
Alpha: 0.001 Lmabda: 59636.23316594637 R2: nan


C:\Users\Gaurish Garg\AppData\Local\Temp\ipykernel_14432\1031378910.py:34: RuntimeWarning: overflow encountered in matmul
  shrinkage = np.matmul((identity_mat-alpha*l*d),betas)


Alpha: 0.001 Lmabda: 104811.3134154683 R2: nan
Alpha: 0.001 Lmabda: 184206.99693267164 R2: nan
Alpha: 0.001 Lmabda: 323745.754281764 R2: nan
Alpha: 0.001 Lmabda: 568986.6029018281 R2: nan
Alpha: 0.001 Lmabda: 1000000.0 R2: nan


In [114]:
r2_max_index = np.nanargmax(r2_tuned)
r2_max_index, r2_tuned[r2_max_index]

(0, 0.9884314078800781)

In [115]:
alpha_lambda[0]

1e-06

In [117]:
l = alpha_lambda[0]
alpha = 0.001
x_train = x_tr
y_train = y_tr
mean_vals = {}
std_vals = {}
## standardising
for column in x_train.columns:
    mean = x_train[column].mean()
    std = x_train[column].std()
    mean_vals[column] = mean
    std_vals[column] = std
    x_train[column] = (x_train[column]-mean)/std

##training
betas = np.zeros(x_train.shape[1]+1)
x_tr1 = np.c_[np.ones(len(x_train)),np.array(x_train)]
y_tr1 = np.array(y_train).ravel()
for i in range(0,10000):       
    predictions = np.matmul(x_tr1,betas)
    gradient = np.matmul(np.transpose(x_tr1),predictions-y_tr1)
    identity_mat = np.identity(len(betas))
    d = np.identity(len(betas))
    d[0][0] = 0
    shrinkage = np.matmul((identity_mat-alpha*l*d),betas)
    n = len(x_tr1)
    ##update
    betas = shrinkage - 1/n*alpha*gradient
train_predictions = np.matmul(x_tr1,betas)
errors = train_predictions - y_tr1
r2_score = 1 - np.sum(errors**2)/(np.sum((y_tr1-np.mean(y_tr1))**2))
print("Train_R2: "+str(r2_score))

Train_R2: 0.988454892530652


In [118]:
betas

array([55.29578771,  7.38336818, 17.59376132,  0.82780575,  0.54668018,
       -0.1501665 ,  0.1501665 ])

In [119]:
predictions = np.matmul(x_t,betas)
predictions.shape

(3000,)

In [120]:
errors = predictions - y_t
errors.shape

(3000,)

In [121]:
r2_score = 1 - np.sum(errors**2)/(np.sum((y_t-np.mean(y_t))**2))
r2_score

0.9894268315043784